# Gold Anchors License Nothing — reproducer

A one-notebook demonstration of the instrument behind the paper *Gold Anchors License Nothing:
label-free auditing of LLM judge panels* (Fathom Lab). It installs `styxx`, then shows the two
behaviors that carry the paper: the label-free auditor **prices** an informative panel (with a
covering interval) and **refuses** an uninformative one — `styxx.anchors` never returns a
clipped confident number when the anchors cannot identify the panel.

**Scope law (never violated):** the method can only *void* an eval, never *bless* one. Passing an
audit is necessary, not sufficient.

Full corpora, preregistrations, per-judge verdict receipts, oracle receipts, and OATH certificates
for every number in the paper are in `fathom-lab/styxx`, `papers/anchored-validity/`.

In [ ]:
!pip -q install styxx>=7.26.0
import numpy as np
from styxx import anchors
print('styxx.anchors ready; audit_panel signature:')
import inspect; print(inspect.signature(anchors.audit_panel))

## 1. A correlated panel it can price

Three judges whose errors share a failure factor (the realistic case that breaks Dawid–Skene).
Known-label anchor strata (`neg`, `pos`) let the auditor *observe* the error structure instead of
assuming independence. True labels are used ONLY to score the result below — never fed to the auditor.

In [ ]:
rng = np.random.default_rng(0)
n, J, k, pi = 400, 3, 60, 0.35

def make_panel(err, corr):
    y = (rng.random(n) < pi).astype(int)             # hidden truth
    shared = rng.random(n) < corr                    # correlated failure event
    V = np.stack([np.where((rng.random(n) < err) | shared, 1 - y, y) for _ in range(J)], 1)
    def anch(label):
        return np.stack([np.where(rng.random(k) < err, 1 - label, label) for _ in range(J)], 1)
    return V, anch(0), anch(1), y

V, neg, pos, y = make_panel(err=0.12, corr=0.10)
r = anchors.audit_panel(V, neg, pos, seed=1)
print('verdict     :', r['verdict'])
print('pi (audit)  :', round(r['pi'], 4))
print('interval    :', [round(x, 4) for x in r['ci']])
print('true prev.  :', round(float(y.mean()), 4), '  <- covered:', r['ci'][0] <= y.mean() <= r['ci'][1])
print('scope       :', r['scope'])

## 2. A deaf panel it refuses

Judges that carry no signal (near-random verdicts). A label-free estimator that *always returns a
number* would emit a confident-looking prevalence here. `styxx.anchors` refuses — the enforced-refusal
primitive.

In [ ]:
Vd  = (rng.random((n, J)) < 0.5).astype(int)
negd = (rng.random((k, J)) < 0.5).astype(int)
posd = (rng.random((k, J)) < 0.5).astype(int)
rd = anchors.audit_panel(Vd, negd, posd, seed=1)
print('verdict :', rd['verdict'])       # -> VOID_PANEL__uninformative
print('pi      :', rd['pi'])            # -> None; no clipped confident number
assert rd['verdict'].startswith('VOID'), 'a content-free panel must be refused'
print('\nThe instrument ships with its weaknesses printed on the label: it refuses rather than guess.')

## What the paper adds beyond this demo

On a **real** correlated judge panel (four prompt-personas of Qwen2.5-3B-Instruct) across four
constructed task families, *blatant gold anchors* (verbatim pairs, direct negations) certify the panel
while label-free coverage of the truth is **0/15 in every family** — and the failure is *silent* when
the violation is smooth. **Ladder anchors** drawn from the same generator as the work items repair it,
either restoring calibrated coverage (13/15) or **refusing** (14/15 VOID) when no judge is informative.
A frontier panel was priced exactly at ceiling. All 36,720 labels were re-derived from item text with
zero mismatches; every number is OATH-certified. See `papers/anchored-validity/` in the repo.